# Activations & Gradients, BatchNorm —— 「为什么这样做」详解版

> 这是 `Activations_&_Gradients,_BatchNorm.ipynb`（逐行注释版）的**姊妹版**：
> 代码**完全一样、一字未改**，但每个 cell 前面都加了一段 **「为什么这样写」** 的原理讲解——
> 回答「这行为什么要乘这个系数 / 为什么沿这个维度 / 为什么用 with / 为什么先别运行」。

---

## 这一讲到底在解决什么问题？

前两讲我们「把 MLP 搭出来、能训练」就满足了。但当网络**变深**,会冒出两个隐形杀手:

1. **激活值(前向信号)失控** —— 越往深层走,数值要么越来越大(爆炸)、要么越来越小(消失);喂进 `tanh` 还会**饱和**(贴到 ±1),饱和处梯度≈0,神经元就「死」了、学不动。
2. **梯度(反向信号)失控** —— 同样会随深度指数级放大或缩小,导致深层根本训练不动。

**这一讲的全部技巧,都是为了让「前向激活」和「反向梯度」在深层里保持健康(方差 ~1、不饱和)**:
- **初始化缩放**(静态地调好开局)
- **BatchNorm**(动态地每步强行拉回)
- **四张诊断图**(装上仪表盘,让你能「看见」网络内部)

---

### ⚠️ 关于「别让电脑卡死」（本版同样只加注释、不改你的代码）

你原代码里有两处 `while torch.no_grad():`（应为 `with`），**直接运行会死循环卡死电脑**;还有 `parameters` 里误放了 `b1`/`bnstd_running`（更新时会崩）。这些我都**原样保留**,只在对应 cell 前用 `> ⚠️` 标出「为什么错、正确怎么写、先别运行」。想一键跑通请用 `Activations_BatchNorm_生产案例.ipynb`。

## 0. 导入库

**为什么这几个库**:`torch` 提供张量+自动求导;`F`(functional)里有 `cross_entropy`——它把 softmax 和负对数似然合在一起,数值上比自己手写 `exp/log` 更稳。**为什么这一讲特别依赖 `matplotlib`**:因为本讲的精髓就是**画分布直方图**去「看」激活和梯度健不健康,图就是这一讲的显微镜。

In [ ]:
import torch                       # PyTorch 主库(张量 + 自动求导)
import torch.nn as nn              # 神经网络模块(本讲主要自己手搓,少量用到)
import torch.nn.functional as F    # 函数式接口:cross_entropy / softmax 等
import  matplotlib.pyplot as plt   # 画图(本讲用来画激活/梯度分布直方图)
%matplotlib inline                 # 让图直接嵌在 notebook 里显示

## 1. 读数据 + 建词表

**为什么和前几讲一模一样**:数据管线(读名字、字符↔编号映射、滑动窗口)在 Part 1/2 已经定型,这一讲的重点完全不在数据,而在**网络内部的数值健康**。所以数据部分照搬,把注意力留给后面的初始化和 BatchNorm。

In [ ]:
words=open('names.txt','r').read().splitlines()  # 读 names.txt,按行切成名字列表(约 32033 个)
words[:8]                                        # 看前 8 个,确认读对了

**为什么 `.` 要编号成 0**:它同时充当「名字开始」和「名字结束」的特殊标记。用同一个符号收尾,采样时只要采到 0 就知道该停;编号 0 也方便把它当作「填充上下文」的默认值(下面 `[0]*block_size` 就是全 `.`)。

In [ ]:
chars=sorted(list(set(''.join(words))))          # 所有出现过的字符,去重排序(a..z 共 26 个)
stoi={s:i+1 for i,s in enumerate(chars)}         # 字符->编号: a=1,b=2,...,z=26
stoi['.']=0                                      # '.' 作为起始/结束特殊符,编号 0
itos={i:s for s,i in stoi.items()}               # 反过来:编号->字符,采样时用
vocab_size=len(itos)                             # 词表大小 = 27
block_size=3                                     # 上下文长度:用前 3 个字符预测下一个

## 2. 构造数据集（滑动窗口）

**为什么要滑动窗口**:模型一次只能看 `block_size=3` 个字符去预测第 4 个。一个名字 `emma` 要拆成多条「上下文→目标」样本(`... → e`、`..e → m`、`.em → m` ...),窗口每次右移一格、丢掉最老的字符补上新的。**为什么封装成函数**:因为紧接着要对 训练/验证/测试 三个集合各调一次,封装避免复制粘贴三遍。

In [ ]:
def build_dataset(words):                         # 输入名字列表 -> 输出 (X, Y)
    X,Y=[],[]                                     # X=上下文(每个3个编号), Y=目标字符编号
    for w in words:                               # 逐个名字
        context=[0]*block_size                    # 上下文初始化为 [0,0,0](全是 '.')
        for ch in w+'.':                          # 遍历名字每个字符,末尾补 '.' 表示结束
            ix=stoi[ch]                           # 当前字符编号
            X.append(context)                     # 记录:当前上下文 ->
            Y.append(ix)                          #        当前字符(目标)
            context=context[1:]+[ix]              # 窗口右移一格:丢掉最左,补上当前字符
    X=torch.tensor(X)                             # list -> 张量,形状 (样本数, 3)
    Y=torch.tensor(Y)                             # list -> 张量,形状 (样本数,)
    return X,Y

## 3. 切分 训练/验证/测试（80/10/10）

**为什么要切三份、为什么固定 `seed(42)`**:训练集用来学参数;验证集用来调超参(学习率、隐藏层大小)并**监控过拟合**;测试集只在最后看一次,代表「真实新数据」。固定随机种子是为了**每次切分结果一致、实验可复现**——否则今天和明天的验证 loss 没法对比。

In [ ]:
import  random                                  # 打乱名字顺序用
random.seed(42)                                  # 固定随机种子,保证每次切分一致
random.shuffle(words)                            # 原地打乱名字列表
n1=int(0.8*len(words))                           # 前 80% 的分界点
n2=int(0.9*len(words))                           # 前 90% 的分界点
Xtr,Ytr=build_dataset(words[:n1])                # 训练集(80%)
Xdev,Ydev=build_dataset(words[n1:n2])            # 验证集(10%),调超参用
Xte,Yte=build_dataset(words[n2:])                # 测试集(10%),最后才看一次

---

# 第一部分：手搓一层带 BatchNorm 的 MLP（cell 5~9）

**为什么先手搓、不直接用 `nn.BatchNorm1d`**:Karpathy 的教学法是「先把黑盒拆开,亲手写一遍,理解每个数字怎么来的,再用现成模块」。这一段用裸张量把「初始化缩放」和「BatchNorm」的每一步都摊开给你看。

> 这一部分和第二部分(cell 10 起的模块化版本)是**两个独立小实验**,各自的 `parameters`/`g` 不同,互不影响。

### 3.1 初始化参数 —— 本讲最关键的两处「为什么乘这个系数」

#### (a) 为什么 `W2 * 0.01`、`b2 * 0`（把输出层缩小）

初始时我们**希望模型「什么都不知道」= 对 27 个字符预测接近均匀分布**。均匀分布的交叉熵正好是 `ln(27) ≈ 3.29`。
如果输出层不缩小,初始 logits 就是一堆忽大忽小的随机数,softmax 后某些字符概率被乱拉高,初始 loss 会是**几十**。后果:训练前几百步全在做一件蠢事——把「过度自信的错误 logits」压回接近 0,loss 曲线呈**曲棍球杆**形(先一横,再猛降)。
乘 `0.01` → 初始 logits ≈ 0 → 初始概率≈均匀 → **开局 loss 就 ≈ 3.29,不浪费任何一步**。

#### (b) 为什么 `W1 * (5/3)/sqrt(fan_in)`（tanh 的 Kaiming 初始化）

- **为什么要 `/sqrt(fan_in)`**:矩阵乘 `x@W` 是把 `fan_in=30` 个数加起来,方差会被放大约 30 倍。不除的话 `hpreact` 的 std 会是好几倍,喂进 tanh 直接饱和成 ±1。除以 `sqrt(fan_in)` 恰好把输出方差拉回 ≈ 1。
- **为什么乘 `5/3`**:tanh 是「压缩型」函数,会把方差**变小**。为了让每层进出方差都维持在 1,要乘一个**增益**抵消这种压缩;tanh 的理论增益就是 `5/3≈1.67`。

> ⚠️ **本格有 bug（按你要求不改）**:`parameters` 里放了 `b1` 和 `bnstd_running`。
> **为什么这是错的**:(1) `b1` 因为后面用了 BatchNorm 会被「减均值」整体抵消、且前向根本没用它 → 它拿不到梯度;(2) `bnstd_running` 是**推理用的滑动统计缓冲区**,本就不该被梯度更新。两者的 `.grad` 恒为 `None`,cell 6 更新时 `-lr*None` 会 **`TypeError` 崩溃**。
> **正确写法**:`parameters=[C,W1,W2,b2,bngain,bnbias]`。

In [ ]:
n_embd=10                                         # 每个字符嵌入成 10 维向量
n_hidden=200                                      # 隐藏层神经元个数
g=torch.Generator().manual_seed(2147483647)       # 固定随机种子,保证可复现
C=torch.randn((vocab_size,n_embd),generator=g)    # 嵌入表 (27, 10)
W1=torch.randn((n_embd*block_size),n_hidden,generator=g)* (5/3)/((n_embd * block_size)**0.5)  # 第一层权重 (30,200),tanh 的 Kaiming 初始化:*(5/3)/sqrt(fan_in)
b1=torch.randn(n_hidden,generator=g)* 0.01        # 第一层偏置(有 BN 后其实多余,见上方 ⚠️)
W2=torch.randn((n_hidden,vocab_size),generator=g)* 0.01  # 输出层权重 (200,27),*0.01 缩小 -> 初始 logits 接近 0
b2=torch.randn(vocab_size,generator=g)* 0         # 输出层偏置,*0 = 初始化为全 0
bngain=torch.ones((1,n_hidden))                   # BatchNorm 的缩放 gamma,初始 1(可训练)
bnbias=torch.zeros((1,n_hidden))                  # BatchNorm 的平移 beta,初始 0(可训练)
bnstd_running=torch.ones((1,n_hidden))            # 推理用的滑动 std(缓冲区,不该训练,见上方 ⚠️)
bnmean_running = torch.zeros((1, n_hidden))       # 推理用的滑动 mean(缓冲区,不该训练)
parameters=[C,W1,W2,b1,b2,bngain,bnbias,bnstd_running]  # ⚠️ b1/bnstd_running 不该在此,正确应为 [C,W1,W2,b2,bngain,bnbias]
for p in parameters:p.requires_grad=True          # 开启梯度追踪(训练前必须)

### 3.2 训练循环 + 手写 BatchNorm

#### 为什么要有 BatchNorm？
上面 (a)(b) 是**静态**地把开局调好,但训练几步后权重一变,`hpreact` 的分布又会漂走、重新饱和。BatchNorm 的思路是**动态**的:**每一步都强行把 `hpreact` 归一化回标准正态**,这样无论上游怎么漂,进 tanh 的永远是「乖」分布。

#### 为什么 `mean(0)` / `std(0)` 是沿第 0 维？
第 0 维是 **batch(这一批 32 个样本)**。BatchNorm = 「对每个神经元,统计这批样本在它身上的均值/std,把它归一化」。所以要沿 batch 维(dim=0)统计,`keepdim=True` 保留成 `(1,200)` 方便广播。

#### 为什么归一化后还要 `bngain`(γ) 和 `bnbias`(β)？
强行拉成均值 0、std 1 太死板,可能限制表达力。于是再给两个**可训练**参数,让网络自己决定「要不要再偏移/放大」。**归一化保证稳定,γ/β 还回自由度**。

#### 为什么前向 `embcat@W1` 没有 `+b1`？
因为下一步就要 `- bnmeani`(减均值),任何偏置都会被整体减掉、**完全多余**。所以带 BN 的线性层一律不加偏置。

#### 为什么要维护 `bnmean_running / bnstd_running`（滑动统计）？
训练时有 32 个样本可以现算统计量;但**推理/上线常常一次只来 1 个样本,没法算 batch 统计**。所以训练期间用**指数滑动平均(EMA)**偷偷记下「整体均值/std」,推理时(cell 9)直接拿来用。

> ⚠️⚠️ **本格有「卡死电脑」的 bug（先别直接运行！）**:`while torch.no_grad():` 应为 `with torch.no_grad():`。
> **为什么会卡死**:`torch.no_grad()` 每次都返回一个**真值**对象,`while` 的条件永远为真 → **死循环,电脑卡死**。
> **为什么该用 `with`**:滑动统计只是「记账」,不属于网络前向、不该进计算图、不该有梯度;`with torch.no_grad()` 正是「这段别追踪梯度」的正确写法。要跑请先把这一个词改成 `with`,并按 cell 5 修好 `parameters`。

In [ ]:
max_steps=200                                     # 训练步数(这里只 200 步,很快,不会卡)
bath_size=32                                      # (原文拼写:应为 batch_size)小批量大小 32
lossi=[]                                           # 记录每步 loss(取 log10)
for i in range(max_steps):
    ix=torch.randint(0,Xtr.shape[0],(bath_size,),generator=g)  # 随机抽 32 个样本的下标
    Xb,Yb=Xtr[ix],Ytr[ix]                          # 这一批的输入 Xb 和目标 Yb
    emb=C[Xb]                                       # 查嵌入表 -> (32, 3, 10)
    embcat=emb.view(emb.shape[0],-1)               # 拼平 3 个字符 -> (32, 30)
    hpreact=embcat@W1                              # 第一层线性(注意没加 b1,因为 BN 会抵消)
    bnmeani=hpreact.mean(0,keepdim=True)           # 沿 batch 维求这一批的均值 (1,200)
    bnstdi=hpreact.std(0,keepdim=True)             # 沿 batch 维求这一批的标准差 (1,200)
    hpreact=bngain*(hpreact-bnmeani)/bnstdi+bnbias # BatchNorm:归一化后再 gamma 缩放 + beta 平移
    with torch.no_grad():
        bnmean_running=0.999*bnmean_running+0.001*bnmeani  # 滑动更新推理用均值(EMA)
        bnstd_running=0.999*bnstd_running+0.001*bnstdi     # 滑动更新推理用标准差(EMA)
    h=torch.tanh(hpreact)                          # tanh 激活 -> (32, 200)
    logits=h@W2+b2                                 # 输出层 -> (32, 27) 的得分
    loss=F.cross_entropy(logits,Yb)                # 交叉熵损失(内部 = softmax + 负对数似然)
    for p in parameters:                           # 清零上一步的梯度
        p.grad=None
    loss.backward()                                # 反向传播,填充各参数的 .grad
    lr=0.1 if i <100 else 0.01                     # 简单的学习率衰减:前 100 步 0.1,之后 0.01
    for p in  parameters:                          # 手写梯度下降更新
        p.data+=-lr*p.grad                         # ⚠️ 若 parameters 含 b1/bnstd_running,这里 p.grad=None 会崩
    if i%10000==0:                                 # 偶尔打印一下进度
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())              # 记录 log10(loss)

### 3.3 画 loss 曲线

**为什么记录的是 `loss.log10()` 而不是 `loss`**:训练后期 loss 变化很小,线性坐标下看着像一条平线、看不出还在不在降。取 log10 能把「小数点后的进步」放大,曲线更能反映真实的下降趋势。

In [ ]:
plt.plot(lossi)                                   # loss(log10) 随步数变化,总体应下降

### 3.4 （可选）训练后一次性标定 BN 统计量

**为什么还有这一格**:除了 cell 6 的「边训练边滑动平均」,还有另一种等价做法——**训练完后把整个训练集过一遍,直接算 `hpreact` 的均值/std** 当推理统计量。Karpathy 证明两种结果几乎一样。这一格是让你对比这个「更直接」的思路。

> ⚠️⚠️ **同样的「卡死电脑」bug**:首行 `while torch.no_grad():` 应为 `with`,否则**死循环卡死**。先改这一个词再运行。

In [ ]:
with torch.no_grad():                            # ⚠️BUG:应为 with!否则死循环卡死电脑
    emb=C[Xtr]                                     # 整个训练集查嵌入
    embcat=emb.view(emb.shape[0],-1)              # 拼平 -> (N, 30)
    hpreact=embcat@W1                             # 第一层线性
    bnmean = hpreact.mean(0, keepdim=True)        # 整个训练集上的均值(一次性标定)
    bnstd = hpreact.std(0, keepdim=True)          # 整个训练集上的标准差

### 3.5 评估 训练/验证 loss

**为什么这里用 `bnmean_running/bnstd_running` 而不是现算的 batch 统计**:这正是 BatchNorm **训练和推理行为不同**的关键。评估/推理时不该依赖「碰巧这一批」的统计量(尤其 batch 很小或只有 1 条时会失真),而要用训练期间累积的、代表**整体**的滑动统计量。**为什么用 `@torch.no_grad()` 装饰器**:评估不需要反向传播,关掉梯度追踪能省内存、更快。

In [ ]:
@torch.no_grad()                                  # 评估不需要梯度,省内存也更快
def split_loss(split):
    x,y={                                          # 按名字选出对应的数据集
        'train':(Xtr,Ytr),
        'val':(Xdev,Ydev),
        'test':(Xte,Yte)
    }[split]
    emb=C[x]                                        # 查嵌入
    embcat=emb.view(emb.shape[0],-1)               # 拼平
    hpreact=embcat@W1                              # 第一层线性
    hpreact=bngain*(hpreact-bnmean_running)/bnstd_running+bnbias  # BN:推理用滑动统计量(不是现算的)
    h=torch.tanh(hpreact)                          # 激活
    logits=h@W2+b2                                 # 输出层
    loss=F.cross_entropy(logits,y)                 # 交叉熵
    print(split,loss.item())                       # 打印该集合的 loss
split_loss('train')                                # 训练集 loss
split_loss('val')                                  # 验证集 loss(和训练集接近说明没过拟合)

---

# 第二部分：模块化 + 搭 6 层深网络看「诊断图」（cell 10~18）

**为什么要模块化、为什么要搭到 6 层**:第一部分只有一层,看不出「深度」带来的问题。要观察激活/梯度如何随**深度**漂移,就得堆很深;而手搓 6 层会是一大坨重复代码。于是先把层封装成**像 PyTorch 那样的类**,再用循环堆深,然后装上「诊断仪表盘」。

### 4.1 把层封装成 Linear / BatchNorm1d / Tanh

**为什么每个类都要有 `__call__` 和 `parameters()`**:这正是 `torch.nn` 的设计——**每个模块自己负责「前向怎么算」和「我有哪些可训练参数」**。有了统一接口,前向就能写成 `for layer in layers: x = layer(x)`,收集参数写成一个列表推导,堆多少层都不怕。

**为什么 `BatchNorm1d` 里要有 `self.training` 开关**:因为 BN 训练时用 batch 统计、推理时用 running 统计,**同一段代码要有两种行为**,用一个布尔开关切换。(这里 `running_mean/var` 的更新正确地用了 `with torch.no_grad()`——可以和 cell 6 的错误 `while` 对照着看。)

In [ ]:
class Linear:                                     # 全连接层:out = x @ W (+ b)
    def __init__(self,fan_in,fan_out,bias=True):
        self.weight=torch.randn((fan_in,fan_out),generator=g)/fan_in**0.5  # Kaiming 初始化:除以 sqrt(fan_in) 保持方差
        self.bias=torch.zeros(fan_out) if bias else None                   # 偏置(可选;配 BN 时通常 bias=False)
    def __call__(self, x):
        self.out=x@self.weight                     # 线性变换
        if self.bias is not None:self.out+=self.bias  # 加偏置(若有)
        return self.out
    def parameters(self):
        return [self.weight]+([] if self.bias is None else [self.bias])  # 返回可训练参数

class BatchNorm1d:                                # 批归一化(1 维)
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps=eps                               # 防止除 0 的小常数
        self.momentum=momentum                     # 滑动统计的动量
        self.training = True                       # 训练模式开关(推理前要置 False)
        self.gamma=torch.ones(dim)                 # 可训练缩放(初始 1)
        self.beta=torch.zeros(dim)                 # 可训练平移(初始 0)
        self.running_mean = torch.zeros(dim)       # 推理用滑动均值(缓冲区)
        self.running_var = torch.ones(dim)         # 推理用滑动方差(缓冲区)
    def __call__(self, x):
        if self.training:                          # 训练:用当前 batch 的统计量
            xmean = x.mean(0, keepdim=True)        # batch 均值
            xvar=x.var(0,keepdim=True)             # batch 方差
        else:                                      # 推理:用滑动统计量
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)  # 归一化到均值0方差1
        self.out = self.gamma * xhat + self.beta   # 再缩放平移
        if self.training:                          # 训练时顺便更新滑动统计量
             with torch.no_grad():                 # (这里正确用了 with;别更新到计算图里)
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum *xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        return self.out
    def parameters(self):
        return [self.gamma, self.beta]             # 只有 gamma/beta 可训练(running_* 是缓冲区)
class Tanh:                                        # 激活层
  def __call__(self, x):
    self.out = torch.tanh(x)                       # tanh 挤到 (-1,1)
    return self.out
  def parameters(self):
    return []                                      # 无可训练参数

### 4.2 搭 6 层深网络

**为什么 `Linear(..., bias=False)`**:每个 Linear 后面都跟 BatchNorm,偏置会被减均值抵消,加了也白加,索性关掉。
**为什么最后一层 `gamma *= 0.1`**:和第一部分「`W2*0.01`」同一个道理——让**初始 logits 更小、更不自信**,初始 loss 更贴近 `ln(27)`,避免曲棍球杆开局。
**为什么留着 `layer.weight *= 1.0 # 5/3` 这句**:这是 Karpathy 故意留的**实验开关**。改成 `5/3` 就是给 tanh 加增益,你可以跑完 cell 13 对比「加不加增益」时激活分布(饱和程度)的差别——这是理解 `5/3` 从哪来的最好方式。

In [ ]:
n_embd = 10 # the dimensionality of the character embedding vectors   # 嵌入维度
n_hidden = 100 # the number of neurons in the hidden layer of the MLP  # 每个隐藏层宽度
g = torch.Generator().manual_seed(2147483647) # for reproducibility    # 固定种子

C = torch.randn((vocab_size, n_embd),            generator=g)          # 嵌入表 (27,10)
layers = [                                                             # 6 层:每层 Linear+BN+Tanh,最后一层只有 Linear+BN
  Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size),
]


with torch.no_grad():                                                  # 初始化微调(不进计算图)
  # last layer: make less confident
  layers[-1].gamma *= 0.1                                              # 最后一层 BN 的 gamma 缩小 -> logits 更小更谦虚
  #layers[-1].weight *= 0.1
  # all other layers: apply gain
  for layer in layers[:-1]:
    if isinstance(layer, Linear):
      layer.weight *= 1.0 #5/3                                         # 实验开关:改成 5/3 给 tanh 加增益,对比激活分布

parameters = [C] + [p for layer in layers for p in layer.parameters()]  # 汇总所有可训练参数
print(sum(p.nelement() for p in parameters)) # number of parameters in total  # 参数总量
for p in parameters:
  p.requires_grad = True                                              # 开启梯度追踪

### 4.3 训练（同时记录 update:data 比例）

**为什么名义 `max_steps=200000` 却在末尾 `if i>=1000: break`**:这一格的目的**不是训练出好模型,而是收集诊断统计量**(下面四张图)。跑 1000 步就足够看清激活/梯度健不健康。这个 break 让你**几秒出图、不用等十几分钟,更不会卡机**。真要训练到底才删掉它(那会跑满 20 万步,耗时几分钟,慢但不卡)。

**为什么每步都 `layer.out.retain_grad()`**:PyTorch 默认只保留**叶子节点(参数)**的梯度,中间层激活的 `.grad` 反向后立刻释放。但诊断图②要画「各层激活的梯度分布」,必须显式 `retain_grad()` 把它们留住。这是纯调试用的,真跑生产会去掉。

**为什么记录 `ud`(update:data 比例)**:见下面诊断图④——它是比「盯 loss」更灵敏的**学习率体检指标**。

In [ ]:
# same optimization as last time
max_steps = 200000                                                    # 名义步数(下面有 break,实际只跑 1001 步)
batch_size = 32                                                       # 小批量大小
lossi = []                                                            # 记录 loss
ud = []                                                               # 记录 update:data 比例(每步每参数)

for i in range(max_steps):

  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)     # 随机抽一批下标
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y                               # 这一批的 X,Y

  # forward pass
  emb = C[Xb] # embed the characters into vectors                     # 查嵌入 (32,3,10)
  x = emb.view(emb.shape[0], -1) # concatenate the vectors            # 拼平 (32,30)
  for layer in layers:                                                # 依次过 6 层
    x = layer(x)
  loss = F.cross_entropy(x, Yb) # loss function                       # 交叉熵损失

  # backward pass
  for layer in layers:
    layer.out.retain_grad() # AFTER_DEBUG: would take out retain_graph # 保留每层输出的梯度(为了画诊断图)
  for p in parameters:
    p.grad = None                                                     # 清零梯度
  loss.backward()                                                     # 反向传播

  # update
  lr = 0.1 if i < 150000 else 0.01 # step learning rate decay          # 学习率衰减
  for p in parameters:
    p.data += -lr * p.grad                                            # 梯度下降更新

  # track stats
  if i % 10000 == 0: # print every once in a while                    # 偶尔打印进度
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())                                   # 记录 log10(loss)
  with torch.no_grad():
    ud.append([((lr*p.grad).std() / p.data.std()).log10().item() for p in parameters])  # 记录 update:data 比例

  if i >= 1000:
    break # AFTER_DEBUG: would take out obviously to run full optimization  # ✅ 只跑 1001 步就停,防卡机

### 诊断图 ①：每层激活值分布

**为什么要看这个、`saturated` 是什么意思**:`saturated = |激活|>0.97 的比例`。tanh 一旦贴到 ±1 就**饱和**,那里的导数≈0,梯度传不回去,这些神经元**学不动、近似「死」了**。健康的分布应该比较「胖」、铺得开,而不是全挤在 ±1。**为什么排除输出层**(`layers[:-1]`):输出层后面没有 tanh,不存在饱和问题。

In [ ]:
plt.figure(figsize=(20, 4)) # width and height of the plot          # 宽 20 高 4 的画布
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer  # 遍历各层(去掉输出层)
  if isinstance(layer, Tanh):                                         # 只看 Tanh 层
    t = layer.out                                                     # 该层的激活值
    print('layer %d (%10s): mean %+.2f, std %.2f, saturated: %.2f%%' % (i, layer.__class__.__name__, t.mean(), t.std(), (t.abs() > 0.97).float().mean()*100))  # 打印均值/标准差/饱和比例
    hy, hx = torch.histogram(t, density=True)                         # 算直方图
    plt.plot(hx[:-1].detach(), hy.detach())                           # 画分布曲线
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends)
plt.title('activation distribution')                                  # 标题:激活分布

### 诊断图 ②：每层激活的梯度分布

**为什么要看这个**:这是从**反向**视角体检。理想情况下,各层的梯度分布应该**大致重合、宽度相近**——说明梯度在深层间传播得很均匀。**如果越深越窄** = 梯度消失(深层学不到);**越深越宽** = 梯度爆炸(不稳)。BatchNorm 的一大好处就是让这些分布保持一致。

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out.grad                                                # 该层激活的梯度(cell12 里 retain_grad 才有)
    print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends)
plt.title('gradient distribution')                                    # 标题:梯度分布

### 诊断图 ③：每个权重矩阵的梯度分布 + grad:data 比例

**为什么看 `grad:data ratio = grad.std()/data.std()`**:它衡量「这一步的梯度,相对权重本身有多大」。**为什么重要**:如果某个权重的这个比例特别大,说明它每步被改动得比别的权重猛得多,训练容易不稳、各层学习速度不均衡。理想是各层这个比例都在同一量级。**为什么只看 `p.ndim==2`**:只关心二维权重矩阵,跳过一维的偏置/γ/β。

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i,p in enumerate(parameters):
  t = p.grad                                                          # 该参数的梯度
  if p.ndim == 2:                                                     # 只看二维权重矩阵(跳过偏置/gamma/beta)
    print('weight %10s | mean %+f | std %e | grad:data ratio %e' % (tuple(p.shape), t.mean(), t.std(), t.std() / p.std()))  # 打印 grad:data 比例
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'{i} {tuple(p.shape)}')
plt.legend(legends)
plt.title('weights gradient distribution');                          # 标题:权重梯度分布

### 诊断图 ④：update:data 比例随训练变化（学习率体检）

**为什么这个比盯 loss 更好用**:loss 抖动大、还受 batch 随机性影响,很难判断学习率合不合适。而「每步参数被改动了自身的百分之几」(`std(lr·grad)/std(param)` 的 log10)是个**稳定、可解释**的量。经验上健康值约 **1e-3(即图上的 -3)**:
- 曲线**远高于 -3** → 学习率太大,每步动得太猛,容易发散;
- 曲线**远低于 -3** → 学习率太小,学得太慢、浪费时间。

黑色参考线画在 -3,让你一眼看出各参数是否落在健康区间。

In [ ]:
plt.figure(figsize=(20, 4))
legends = []
for i,p in enumerate(parameters):
  if p.ndim == 2:                                                     # 只看二维权重
    plt.plot([ud[j][i] for j in range(len(ud))])                      # 画该参数每步的 update:data 比例
    legends.append('param %d' % i)
plt.plot([0, len(ud)], [-3, -3], 'k') # these ratios should be ~1e-3, indicate on plot  # 参考线:1e-3
plt.legend(legends);

### 4.4 评估（切换到推理模式）

**为什么评估前一定要 `layer.training = False`**:不切的话,BatchNorm 仍会用「当前这一批」的统计量。而这里评估传入的是**整个** Xtr/Xdev(几万条),或线上**逐条**推理(1 条)——用「这一批」的统计量既不代表整体、batch=1 时还会直接出错。切成 `False` 后 BN 改用训练期间累积的 `running_mean/var`,才是正确的推理行为。**这是 BN 最容易踩的生产坑**,配套的 `生产案例` 笔记本案例 5 专门演示了它。

In [ ]:
@torch.no_grad() # this decorator disables gradient tracking          # 评估不追踪梯度
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)                                # 查嵌入
  x = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd) # 拼平
  for layer in layers:                                                # 过所有层
    x = layer(x)
  loss = F.cross_entropy(x, y)                                        # 交叉熵
  print(split, loss.item())

# put layers into eval mode
for layer in layers:
  layer.training = False                                             # 关键:切到推理模式(BN 用 running 统计量)
split_loss('train')
split_loss('val')

### 4.5 从模型采样生成名字

**为什么这样采样**:从起始上下文 `[0,0,0]`(全 `.`)开始,前向得到 27 个字符的概率,用 `torch.multinomial` **按概率随机采**一个(而不是每次都取最大——那样只会生成一个固定名字)。采完把窗口右移、把新字符接上,直到采到 `.`(编号 0)表示名字结束。**为什么此时结果才「像样」**:因为前一格已把网络切到 `training=False`,BN 用的是稳定的 running 统计量。

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)                    # 采样用的随机种子(+10 换一批结果)

for _ in range(20):                                                   # 生成 20 个名字

    out = []
    context = [0] * block_size # initialize with all ...              # 初始上下文 [0,0,0]
    while True:
      # forward pass the neural net
      emb = C[torch.tensor([context])] # (1,block_size,n_embd)        # 当前上下文的嵌入
      x = emb.view(emb.shape[0], -1) # concatenate the vectors        # 拼平
      for layer in layers:                                           # 过网络(此时应已 training=False)
        x = layer(x)
      logits = x
      probs = F.softmax(logits, dim=1)                               # 转成概率分布
      # sample from the distribution
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()  # 按概率采样下一个字符
      # shift the context window and track the samples
      context = context[1:] + [ix]                                   # 窗口右移
      out.append(ix)
      # if we sample the special '.' token, break
      if ix == 0:                                                    # 采到 '.' 就结束这个名字
        break

    print(''.join(itos[i] for i in out)) # decode and print the generated word  # 编号转回字符并打印

---

## 附录 A：BatchNorm 前向的交互式演示（滑块）

**为什么放这个小工具**:用来建立**直觉**——拖动滑块改变某个样本的值,你会看到无论输入(蓝点)分布多离谱,BatchNorm 都把它们减均值除标准差、**映射回标准正态**(红点)。一句话:BN 就是「把任意分布强行拉回均值 0、std 1」。

> 需要 `ipywidgets`+`scipy`(`pip install ipywidgets scipy`)。运行安全、不会卡机。

In [ ]:
# BatchNorm forward pass as a widget
from ipywidgets import interact, interactive, fixed, interact_manual   # 交互滑块
import ipywidgets as widgets
import scipy.stats as stats                                            # 画正态曲线
import numpy as np

def normshow(x0):

  g = torch.Generator().manual_seed(2147483647+1)
  x = torch.randn(5, generator=g) * 5                                  # 5 个样本,乘 5 拉大方差
  x[0] = x0 # override the 0th example with the slider                 # 用滑块覆盖第 0 个样本
  mu = x.mean()                                                        # 这批的均值
  sig = x.std()                                                        # 这批的标准差
  y = (x - mu)/sig                                                     # BN 归一化结果

  plt.figure(figsize=(10, 5))
  # plot 0
  plt.plot([-6,6], [0,0], 'k')
  # plot the mean and std
  xx = np.linspace(-6, 6, 100)
  plt.plot(xx, stats.norm.pdf(xx, mu, sig), 'b')                       # 蓝:输入分布
  xx = np.linspace(-6, 6, 100)
  plt.plot(xx, stats.norm.pdf(xx, 0, 1), 'r')                          # 红:标准正态(BN 目标)
  # plot little lines connecting input and output
  for i in range(len(x)):
    plt.plot([x[i],y[i]], [1, 0], 'k', alpha=0.2)                      # 连线:输入 -> 输出
  # plot the input and output values
  plt.scatter(x.data, torch.ones_like(x).data, c='b', s=100)          # 蓝点:输入
  plt.scatter(y.data, torch.zeros_like(y).data, c='r', s=100)         # 红点:归一化输出
  plt.xlim(-6, 6)
  # title
  plt.title('input mu %.2f std %.2f' % (mu, sig))

interact(normshow, x0=(-30,30,0.5));                                   # 生成滑块(-30~30)

## 附录 B：纯 Linear 链的前向/反向统计

**为什么做这个实验**:亲眼看看「**不做任何归一化**时,方差会怎样漂移」。这里 `b @ a` 的权重没有除 `sqrt(fan_in)`,所以你会看到前向输出 `c` 的 std 被明显放大——这正是前面为什么要做 Kaiming 缩放的实证。**为什么要 `retain_grad()`**:`a/b/c` 里有的是非叶子节点,不 retain 的话反向后 `.grad` 会被释放、打印不出来。

In [ ]:
# Linear: activation statistics of forward and backward pass

g = torch.Generator().manual_seed(2147483647)

a = torch.randn((1000,1), requires_grad=True, generator=g)          # a.grad = b.T @ c.grad     # 输入向量
b = torch.randn((1000,1000), requires_grad=True, generator=g)       # b.grad = c.grad @ a.T     # 权重矩阵(未缩放)
c = b @ a                                                            # 前向:矩阵乘
loss = torch.randn(1000, generator=g) @ c                           # 随便造一个标量 loss
a.retain_grad()                                                     # 保留非叶子节点梯度以便观察
b.retain_grad()
c.retain_grad()
loss.backward()                                                    # 反向传播
print('a std:', a.std().item())                                    # 看前向各量的标准差
print('b std:', b.std().item())
print('c std:', c.std().item())                                    # 注意 c 的 std 被放大了(未除 sqrt(fan_in))
print('-----')
print('c grad std:', c.grad.std().item())                          # 看反向各量梯度的标准差
print('a grad std:', a.grad.std().item())
print('b grad std:', b.grad.std().item())

## 附录 C：Linear + BatchNorm 的前向/反向统计

**为什么和附录 B 对照着看**:附录 B 里方差乱漂,这里在 Linear 后面加一层 BatchNorm,你会看到前向 `out` 的 std 被**稳稳按在 ~1**,反向梯度也更「守规矩」。**这就是 BatchNorm 让深层网络好训练的直接证据**——它把「方差随深度失控」这个根本问题按住了。

---

### 全篇「为什么」一句话总结
Part 3 每一处看似奇怪的写法,都在回答同一个问题:**怎么让前向激活和反向梯度在深层网络里既不爆炸也不消失**。
- **初始化缩放** = 静态调好开局;
- **BatchNorm** = 动态每步拉回;
- **四张诊断图 + ud 比例** = 仪表盘,让你能「看见」而不是瞎调。

In [ ]:
# Linear + BatchNorm: activation statistics of forward and backward pass

g = torch.Generator().manual_seed(2147483647)

n = 1000
# linear layer ---
inp = torch.randn(n, requires_grad=True, generator=g)               # 输入
w = torch.randn((n, n), requires_grad=True, generator=g) # / n**0.5  # 权重(注释里的 /n**0.5 是可选的缩放对比)
x = w @ inp                                                         # 线性输出
# bn layer ---
xmean = x.mean()                                                   # 均值
xvar = x.var()                                                     # 方差
out = (x - xmean) / torch.sqrt(xvar + 1e-5)                        # BN 归一化 -> std≈1
# ----
loss = out @ torch.randn(n, generator=g)                           # 造一个标量 loss
inp.retain_grad()
x.retain_grad()
w.retain_grad()
out.retain_grad()
loss.backward()                                                    # 反向传播

print('inp std: ', inp.std().item())                               # 前向各量 std
print('w std: ', w.std().item())
print('x std: ', x.std().item())
print('out std: ', out.std().item())                               # BN 后 out 的 std 稳定在 ~1
print('------')
print('out grad std: ', out.grad.std().item())                     # 反向各量梯度 std
print('x grad std: ', x.grad.std().item())
print('w grad std: ', w.grad.std().item())
print('inp grad std: ', inp.grad.std().item())